<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보충 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Tiktoken BPE 토크나이저에 새로운 토큰 확장하기 (Extending the Tiktoken BPE Tokenizer with New Tokens)

- 이 노트북은 기존 BPE 토크나이저를 확장하는 방법을 설명합니다. 특히, 인기 있는 [tiktoken](https://github.com/openai/tiktoken) 구현에 대해 어떻게 수행하는지에 중점을 둡니다
- 토큰화에 대한 일반적인 소개는 [2장](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)과 BPE from Scratch [link] 튜토리얼을 참조하세요
- 예를 들어, GPT-2 토크나이저가 있고 다음 텍스트를 인코딩하려고 한다고 가정해보겠습니다:

In [ ]:
import tiktoken

base_tokenizer = tiktoken.get_encoding("gpt2")
sample_text = "Hello, MyNewToken_1 is a new token. <|endoftext|>"

token_ids = base_tokenizer.encode(sample_text, allowed_special={"<|endoftext|>"})
print(token_ids)

- 각 토큰 ID를 반복하면 토큰 ID가 어휘를 통해 어떻게 디코딩되는지 더 잘 이해할 수 있습니다:

In [ ]:
for token_id in token_ids:
    print(f"{token_id} -> {base_tokenizer.decode([token_id])}")

- 위에서 볼 수 있듯이, `"MyNewToken_1"`이 5개의 개별 하위단어 토큰으로 분해됩니다 -- 이는 BPE가 알려지지 않은 단어를 처리할 때의 정상적인 동작입니다
- 하지만 다른 단어들이나 `"<|endoftext|>"`와 유사하게 단일 토큰으로 인코딩하고 싶은 특수 토큰이라고 가정해보겠습니다; 이 노트북은 그 방법을 설명합니다

&nbsp;
## 1. 특수 토큰 추가하기 (Adding special tokens)

- 새로운 토큰을 특수 토큰으로 추가해야 한다는 점을 주목하세요. 그 이유는 토크나이저 훈련 과정에서 생성되는 새로운 토큰에 대한 "병합(merges)"이 없기 때문입니다 -- 병합이 있더라도 기존 토큰화 체계를 깨뜨리지 않고 통합하는 것은 매우 어려울 것입니다 ("병합"을 이해하려면 BPE from scratch 노트북 [link]을 참조하세요)
- 2개의 새로운 토큰을 추가하려고 한다고 가정해보겠습니다:

In [ ]:
# 사용자 정의 토큰과 해당 토큰 ID 정의
custom_tokens = ["MyNewToken_1", "MyNewToken_2"]
custom_token_ids = {
    token: base_tokenizer.n_vocab + i for i, token in enumerate(custom_tokens)
}

- 다음으로, 다음과 같이 특수 토큰을 보유하는 사용자 정의 `Encoding` 객체를 생성합니다:

In [ ]:
# 확장된 토큰으로 새로운 Encoding 객체 생성
extended_tokenizer = tiktoken.Encoding(
    name="gpt2_custom",
    pat_str=base_tokenizer._pat_str,
    mergeable_ranks=base_tokenizer._mergeable_ranks,
    special_tokens={**base_tokenizer._special_tokens, **custom_token_ids},
)

- 이제 완료되었습니다. 이제 샘플 텍스트를 인코딩할 수 있는지 확인할 수 있습니다:

- 볼 수 있듯이, 새로운 토큰 `50257`과 `50258`이 이제 출력에 인코딩됩니다:

In [ ]:
special_tokens_set = set(custom_tokens) | {"<|endoftext|>"}

token_ids = extended_tokenizer.encode(
    "Sample text with MyNewToken_1 and MyNewToken_2. <|endoftext|>",
    allowed_special=special_tokens_set
)
print(token_ids)

- 다시, 토큰별로도 살펴볼 수 있습니다:

In [ ]:
for token_id in token_ids:
    print(f"{token_id} -> {extended_tokenizer.decode([token_id])}")

- 위에서 볼 수 있듯이, 토크나이저를 성공적으로 업데이트했습니다
- 하지만 사전훈련된 LLM과 함께 사용하려면, LLM의 임베딩 및 출력 층도 업데이트해야 합니다. 이는 다음 섹션에서 논의됩니다

&nbsp;
## 2. 사전훈련된 LLM 업데이트하기 (Updating a pretrained LLM)

- 이 섹션에서는 토크나이저를 업데이트한 후 기존 사전훈련된 LLM을 어떻게 업데이트해야 하는지 살펴보겠습니다
- 이를 위해 메인 책에서 사용되는 원래 사전훈련된 GPT-2 모델을 사용합니다

&nbsp;
### 2.1 사전훈련된 GPT 모델 로딩하기 (Loading a pretrained GPT model)

In [ ]:
from llms_from_scratch.ch05 import download_and_load_gpt2
# llms_from_scratch 설치 지침은 다음을 참조하세요:
# https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg

settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
from llms_from_scratch.ch04 import GPTModel
# llms_from_scratch 설치 지침은 다음을 참조하세요:
# https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # 어휘 크기
    "context_length": 256, # 축약된 컨텍스트 길이 (원본: 1024)
    "emb_dim": 768,        # 임베딩 차원
    "n_heads": 12,         # 어텐션 헤드 수
    "n_layers": 12,        # 레이어 수
    "drop_rate": 0.1,      # 드롭아웃 비율
    "qkv_bias": False      # Query-key-value 편향
}

# 간결성을 위해 딕셔너리에서 모델 구성 정의
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# 기본 구성을 복사하고 특정 모델 설정으로 업데이트
model_name = "gpt2-small (124M)"  # 예제 모델명
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval();

### 2.2 사전훈련된 GPT 모델 사용하기 (Using the pretrained GPT model)

- 다음으로, 원래 토크나이저와 새로운 토크나이저를 사용하여 토큰화하는 아래의 샘플 텍스트를 고려해보겠습니다:

In [ ]:
sample_text = "Sample text with MyNewToken_1 and MyNewToken_2. <|endoftext|>"

original_token_ids = base_tokenizer.encode(
    sample_text, allowed_special={"<|endoftext|>"}
)

In [ ]:
new_token_ids = extended_tokenizer.encode(
    "Sample text with MyNewToken_1 and MyNewToken_2. <|endoftext|>",
    allowed_special=special_tokens_set
)

- 이제 원래 토큰 ID를 GPT 모델에 전달해보겠습니다:

In [ ]:
import torch

with torch.no_grad():
    out = gpt(torch.tensor([original_token_ids]))

print(out)

- 위에서 볼 수 있듯이, 이는 문제없이 작동합니다 (코드는 간단함을 위해 출력을 텍스트로 다시 변환하지 않고 원시 출력을 보여줍니다. 자세한 내용은 5장 [link] 섹션 5.3.3의 `generate` 함수를 확인하세요)

- 이제 업데이트된 토크나이저가 생성한 토큰 ID로 동일한 시도를 하면 어떻게 될까요?

```python
with torch.no_grad():
    gpt(torch.tensor([new_token_ids]))

print(out)

...
# IndexError: index out of range in self
```

- 볼 수 있듯이, 이는 인덱스 오류를 발생시킵니다
- 그 이유는 GPT 모델이 입력 임베딩 층과 출력 층을 통해 고정된 어휘 크기를 기대하기 때문입니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/extend-tiktoken/gpt-updates.webp" width="400px">

&nbsp;
### 2.3 임베딩 층 업데이트하기 (Updating the embedding layer)

- 임베딩 층 업데이트부터 시작해보겠습니다
- 먼저, 임베딩 층이 어휘 크기에 해당하는 50,257개의 항목을 가지고 있음을 주목하세요:

In [ ]:
gpt.tok_emb

- 이 임베딩 층을 2개의 항목을 더 추가하여 확장하려고 합니다
- 간단히 말해서, 더 큰 크기의 새로운 임베딩 층을 생성하고, 그다음 이전 임베딩 층 값들을 복사합니다

In [ ]:
num_tokens, emb_size = gpt.tok_emb.weight.shape
new_num_tokens = num_tokens + 2

# 새로운 임베딩 층 생성
new_embedding = torch.nn.Embedding(new_num_tokens, emb_size)

# 이전 임베딩 층에서 가중치 복사
new_embedding.weight.data[:num_tokens] = gpt.tok_emb.weight.data

# 모델의 이전 임베딩 층을 새로운 것으로 교체
gpt.tok_emb = new_embedding

print(gpt.tok_emb)

- 위에서 볼 수 있듯이, 이제 증가된 임베딩 층을 가지고 있습니다

&nbsp;
### 2.4 출력 층 업데이트하기 (Updating the output layer)

- 다음으로, 임베딩 층과 유사하게 어휘 크기에 해당하는 50,257개의 출력 특성을 가진 출력 층을 확장해야 합니다 (참고로, PyTorch에서 Linear 층과 Embedding 층 사이의 유사성을 논의하는 보너스 자료가 유용할 것입니다)

In [ ]:
gpt.out_head

- 출력 층을 확장하는 절차는 임베딩 층을 확장하는 것과 유사합니다:

In [ ]:
original_out_features, original_in_features = gpt.out_head.weight.shape

# 새로운 출력 특성 수 정의 (예: 2개의 새로운 토큰 추가)
new_out_features = original_out_features + 2

# 확장된 출력 크기로 새로운 선형 층 생성
new_linear = torch.nn.Linear(original_in_features, new_out_features)

# 원래 선형 층에서 가중치와 편향 복사
with torch.no_grad():
    new_linear.weight[:original_out_features] = gpt.out_head.weight
    if gpt.out_head.bias is not None:
        new_linear.bias[:original_out_features] = gpt.out_head.bias

# 원래 선형 층을 새로운 것으로 교체
gpt.out_head = new_linear

print(gpt.out_head)

- 먼저 원래 토큰 ID에서 이 업데이트된 모델을 시도해보겠습니다:

In [ ]:
with torch.no_grad():
    output = gpt(torch.tensor([original_token_ids]))
print(output)

- 다음으로, 업데이트된 토큰에서 시도해보겠습니다:

In [ ]:
with torch.no_grad():
    output = gpt(torch.tensor([new_token_ids]))
print(output)

- 볼 수 있듯이, 모델이 확장된 토큰 세트에서 작동합니다
- 실제로는 이제 새로운 토큰을 포함하는 데이터에서 모델(특히 새로운 임베딩 및 출력 층)을 미세조정(또는 지속적으로 사전훈련)하려고 합니다

**가중치 공유에 대한 주의사항**

- 모델이 가중치 공유를 사용하는 경우, 즉 임베딩 층과 출력 층이 Llama 3 [link]와 유사하게 동일한 가중치를 공유하는 경우, 출력 층 업데이트가 훨씬 간단합니다
- 이 경우, 임베딩 층에서 가중치를 단순히 복사할 수 있습니다:

In [ ]:
gpt.out_head.weight = gpt.tok_emb.weight

In [ ]:
with torch.no_grad():
    output = gpt(torch.tensor([new_token_ids]))